# Pointwise relative-error challenge after HEALPix-to-latitude–longitude remapping

This notebook reproduces the analysis in `02-relative-error-bound.ipynb` for ground-truth and decoded HEALPix Zarr data. Each selected map is first interpolated onto the challenge's original regular grid: 721 latitudes ordered from $90^\circ$ to $-90^\circ$ and 1440 longitudes from $0^\circ$ to $359.75^\circ$, both spaced by $0.25^\circ$.

The workflow is: **aligned HEALPix maps → common 0.25° grid → original pointwise relative-error check**. A target-grid cell passes when

$$|\hat{x}-x| \leq |x|\,\epsilon_{rel}.$$

This exactly preserves the reference challenge's treatment of zeros: if $x=0$, the decoded value must also be exactly zero. The plotted signed relative error is zero for identical values and $(\hat{x}-x)/|x|$ otherwise. Both fields undergo the same HEALPix interpolation before comparison.

## 1. Environment and imports

Run this notebook in the `climgen` environment. The cell configures writable plotting caches and imports xarray, Healpy, plotting, and progress-bar dependencies. No checkpoint or GPU is required.

In [ ]:
import json
import os
import platform
import re
import warnings
from functools import lru_cache
from pathlib import Path
from urllib.parse import urlsplit

CACHE_ROOT = Path('/tmp') / f'healpix_latlon_relative_cache_{os.getuid()}'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT))

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from tqdm.auto import tqdm

display(pd.Series({
    'Python': platform.python_version(),
    'xarray': xr.__version__,
    'healpy': hp.__version__,
}).to_frame('version'))

## 2. Editable analysis configuration

Provide the original HEALPix Zarr and the decoded Zarr exported by the inference notebook. `VARIABLES` follows the other compression notebooks: timestep ranges are end-exclusive; variables may be grouped under `2D` and `3D`; and `level_indices=None` selects all levels. Delete `timesteps` to use every decoded timestep and omit both variable groups to discover every compatible numeric variable.

`RELATIVE_ERROR_BOUND=0.01` is the original challenge's 1% threshold. Ordering is normally read from metadata; set it explicitly only if the store does not identify NESTED or RING ordering.

In [ ]:
GROUND_TRUTH_ZARR = os.environ.get(
    'HPX_RELATIVE_GROUND_TRUTH', '/path/to/ground_truth_healpix.zarr'
)
DECODED_ZARR = os.environ.get(
    'HPX_RELATIVE_DECODED', '/path/to/decoded_fields.zarr'
)
OUTPUT_DIR = Path(os.environ.get(
    'HPX_LATLON_RELATIVE_OUTPUT', './healpix_to_latlon_relative_error_results'
))

VARIABLES = {
    'timesteps': [0],  # Delete this entry to evaluate all decoded timesteps.
    # With no groups, all common numeric HEALPix variables and levels are used.
    # '2D': {'pr': {}},
    # '3D': {'hus': {'level_indices': [0, 2, 5]}},
}

RELATIVE_ERROR_BOUND = 0.01  # 1%, as in the original challenge.
GROUND_TRUTH_ORDERING = 'auto'  # 'auto', 'NESTED', or 'RING'
DECODED_ORDERING = 'auto'       # 'auto', 'NESTED', or 'RING'

## 3. Dataset selection and alignment

These helpers open local or remote stores, identify HEALPix/time/level dimensions, expand requested index ranges, and align reconstructed samples with their ground truth. The decoded export's `source_time_index` is authoritative when present. Incompatible dimensions or coordinates fail before full maps are read.

In [ ]:
def is_remote(location):
    scheme = urlsplit(str(location)).scheme.lower()
    return bool(scheme and scheme != 'file')


def open_zarr_store(location):
    if not is_remote(location) and not Path(location).expanduser().exists():
        raise FileNotFoundError(f'Zarr store does not exist: {location}')
    return xr.open_zarr(str(location), consolidated=None)


def expand_positions(selection, size, label):
    if selection is None:
        return list(range(size))
    items = selection if isinstance(selection, (list, tuple)) else [selection]
    positions = []
    for item in items:
        if isinstance(item, (int, np.integer)) and not isinstance(item, bool):
            positions.append(int(item))
        elif isinstance(item, str) and (match := re.fullmatch(r'\s*(\d+)\s*-\s*(\d+)\s*', item)):
            start, stop = map(int, match.groups())
            if stop <= start:
                raise ValueError(f'Invalid {label} range {item!r}: stop must exceed start.')
            positions.extend(range(start, stop))
        else:
            raise ValueError(f'Invalid {label} entry {item!r}.')
    if not positions or min(positions) < 0 or max(positions) >= size:
        raise IndexError(f'{label} positions must fall in [0, {size - 1}].')
    if len(positions) != len(set(positions)):
        raise ValueError(f'{label} selection contains duplicates or overlapping ranges.')
    return positions


def normalize_level_positions(selection, size, variable):
    if selection is None:
        return list(range(size))
    items = [selection] if isinstance(selection, (int, np.integer)) else list(selection)
    if not items or any(not isinstance(index, (int, np.integer)) for index in items):
        raise ValueError(f'level_indices for {variable!r} must contain integers.')
    positions = [int(index) % size if int(index) < 0 else int(index) for index in items]
    if min(positions) < 0 or max(positions) >= size:
        raise IndexError(f'level_indices for {variable!r} exceed the decoded depth {size}.')
    if len(positions) != len(set(positions)):
        raise ValueError(f'level_indices for {variable!r} contains duplicates.')
    return positions


def flatten_variable_configuration(configuration):
    entries = {str(key): value for key, value in configuration.items() if str(key) != 'timesteps'}
    if any(group in entries for group in ('2D', '3D')):
        unexpected = set(entries) - {'2D', '3D'}
        if unexpected:
            raise ValueError(f'Unexpected entries beside 2D/3D groups: {sorted(unexpected)}')
        flattened = {}
        for group in ('2D', '3D'):
            for variable, specification in (entries.get(group) or {}).items():
                if variable in flattened:
                    raise ValueError(f'Variable {variable!r} occurs in more than one group.')
                flattened[str(variable)] = specification or {}
        return flattened
    return {str(variable): specification or {} for variable, specification in entries.items()}


def healpix_dimension(data_array):
    candidates = []
    for dimension, size in data_array.sizes.items():
        try:
            nside = hp.npix2nside(int(size))
        except ValueError:
            continue
        if nside > 0 and nside & (nside - 1) == 0:
            priority = 0 if ('cell' in dimension.lower() or 'pixel' in dimension.lower()) else 1
            candidates.append((priority, dimension, nside))
    if not candidates:
        raise ValueError(f'{data_array.name!r} has no valid HEALPix cell dimension.')
    candidates.sort()
    if len(candidates) > 1 and candidates[0][0] == candidates[1][0]:
        raise ValueError(f'Ambiguous HEALPix dimensions for {data_array.name!r}: {candidates}.')
    return candidates[0][1], candidates[0][2]


def axis_dimension(data_array, kind, excluded):
    aliases = {
        'time': {'time', 'times', 'date', 'datetime'},
        'level': {'lev', 'level', 'levels', 'plev', 'height', 'depth', 'altitude', 'model_level'},
    }[kind]
    axis = {'time': 'T', 'level': 'Z'}[kind]
    candidates = []
    for dimension in data_array.dims:
        if dimension in excluded:
            continue
        coordinate = data_array.coords.get(dimension)
        attrs = {} if coordinate is None else {str(k).lower(): str(v).lower() for k, v in coordinate.attrs.items()}
        score = 50 * (dimension.lower() in aliases) + 100 * (attrs.get('axis', '').upper() == axis)
        if kind == 'time':
            score += 100 * (attrs.get('standard_name') == 'time')
        if score:
            candidates.append((score, dimension))
    return max(candidates)[1] if candidates else None


def detect_ordering(dataset, requested, label):
    requested = str(requested).upper()
    if requested in {'NESTED', 'RING'}:
        return requested
    if requested != 'AUTO':
        raise ValueError(f'{label}_ORDERING must be auto, NESTED, or RING.')
    for attributes in [dataset.attrs] + [value.attrs for value in dataset.variables.values()]:
        for key, value in attributes.items():
            key_lower, value_lower = str(key).lower(), str(value).lower()
            if 'order' in key_lower and 'nest' in value_lower:
                return 'NESTED'
            if 'order' in key_lower and 'ring' in value_lower:
                return 'RING'
            if 'nest' in key_lower and isinstance(value, (bool, np.bool_)):
                return 'NESTED' if bool(value) else 'RING'
    warnings.warn(f'No HEALPix ordering metadata found for {label}; assuming NESTED.', stacklevel=2)
    return 'NESTED'


def coordinate_positions(reference, requested, label):
    positions = reference.to_index().get_indexer(pd.Index(np.asarray(requested)))
    if (positions < 0).any():
        missing = np.asarray(requested)[positions < 0]
        raise ValueError(f'{label} coordinates are absent from ground truth: {missing.tolist()}')
    return positions.tolist()

## 4. Prepare aligned HEALPix map pairs

The following functions apply timestep and level selections and yield one aligned ground-truth/reconstruction pair at a time. This prevents full-test remapped fields from accumulating in memory. After selecting one time and one level, every variable must reduce to a one-dimensional HEALPix map.

In [ ]:
def prepare_variable_pair(ground_truth, decoded, variable, specification, timestep_selection):
    if variable not in ground_truth.data_vars or variable not in decoded.data_vars:
        raise KeyError(f'{variable!r} must be a data variable in both stores.')
    truth, reconstruction = ground_truth[variable], decoded[variable]
    truth_cell, truth_nside = healpix_dimension(truth)
    decoded_cell, decoded_nside = healpix_dimension(reconstruction)
    if truth_nside != decoded_nside:
        raise ValueError(f'{variable!r} has NSIDE {truth_nside} in ground truth and {decoded_nside} decoded.')

    decoded_time = axis_dimension(reconstruction, 'time', {decoded_cell})
    truth_time = axis_dimension(truth, 'time', {truth_cell})
    if decoded_time is None:
        if timestep_selection not in (None, [], [0]):
            raise ValueError(f'{variable!r} has no decoded time dimension.')
        selected_times = [0]
    else:
        selected_times = expand_positions(timestep_selection, reconstruction.sizes[decoded_time], 'timestep')
        reconstruction = reconstruction.isel({decoded_time: selected_times})
        if truth_time is None:
            raise ValueError(f'{variable!r} has decoded time but no ground-truth time dimension.')
        if 'source_time_index' in decoded and decoded_time in decoded['source_time_index'].dims:
            source_positions = np.asarray(
                decoded['source_time_index'].isel({decoded_time: selected_times}).values, dtype=int
            )
            if source_positions.min() < 0 or source_positions.max() >= truth.sizes[truth_time]:
                raise IndexError('Decoded source_time_index falls outside the ground-truth store.')
            truth = truth.isel({truth_time: source_positions.tolist()})
        elif decoded_time in reconstruction.coords and truth_time in truth.coords:
            truth = truth.isel({truth_time: coordinate_positions(
                truth[truth_time], reconstruction[decoded_time].values, 'time'
            )})
        elif truth.sizes[truth_time] == decoded.sizes[decoded_time]:
            truth = truth.isel({truth_time: selected_times})
        else:
            raise ValueError('Cannot align decoded and ground-truth time axes.')

    decoded_level = axis_dimension(reconstruction, 'level', {decoded_cell, decoded_time})
    truth_level = axis_dimension(truth, 'level', {truth_cell, truth_time})
    if decoded_level is None:
        if specification.get('level_indices') is not None:
            raise ValueError(f'{variable!r} has no decoded level dimension.')
        selected_levels = [0]
        if truth_level is not None:
            raise ValueError(f'{variable!r} has a ground-truth level dimension but decoded data does not.')
    else:
        selected_levels = normalize_level_positions(
            specification.get('level_indices'), reconstruction.sizes[decoded_level], variable
        )
        reconstruction = reconstruction.isel({decoded_level: selected_levels})
        if truth_level is None:
            raise ValueError(f'{variable!r} has decoded levels but no ground-truth level dimension.')
        if decoded_level in reconstruction.coords and truth_level in truth.coords:
            truth = truth.isel({truth_level: coordinate_positions(
                truth[truth_level], reconstruction[decoded_level].values, 'level'
            )})
        elif truth.sizes[truth_level] == decoded.sizes[decoded_level]:
            truth = truth.isel({truth_level: selected_levels})
        else:
            raise ValueError('Cannot align decoded and ground-truth level axes.')

    if set(truth.dims) - {truth_cell, truth_time, truth_level, None}:
        raise ValueError(f'{variable!r} has unsupported ground-truth dimensions.')
    if set(reconstruction.dims) - {decoded_cell, decoded_time, decoded_level, None}:
        raise ValueError(f'{variable!r} has unsupported decoded dimensions.')
    return {
        'truth': truth, 'decoded': reconstruction, 'truth_time': truth_time,
        'decoded_time': decoded_time, 'truth_level': truth_level,
        'decoded_level': decoded_level, 'time_positions': selected_times,
        'level_positions': selected_levels, 'nside': truth_nside,
    }


def iter_aligned_maps(pair):
    n_times = pair['decoded'].sizes[pair['decoded_time']] if pair['decoded_time'] else 1
    n_levels = pair['decoded'].sizes[pair['decoded_level']] if pair['decoded_level'] else 1
    for time_offset in range(n_times):
        for level_offset in range(n_levels):
            truth_indexers, decoded_indexers = {}, {}
            if pair['truth_time']:
                truth_indexers[pair['truth_time']] = time_offset
                decoded_indexers[pair['decoded_time']] = time_offset
            if pair['truth_level']:
                truth_indexers[pair['truth_level']] = level_offset
                decoded_indexers[pair['decoded_level']] = level_offset
            truth_map = np.asarray(pair['truth'].isel(truth_indexers).values).squeeze()
            decoded_map = np.asarray(pair['decoded'].isel(decoded_indexers).values).squeeze()
            if truth_map.ndim != 1 or decoded_map.ndim != 1:
                raise ValueError(f'Expected one-dimensional HEALPix maps, got {truth_map.shape} and {decoded_map.shape}.')
            yield {
                'truth': truth_map, 'decoded': decoded_map,
                'timestep_position': pair['time_positions'][time_offset],
                'level_position': pair['level_positions'][level_offset],
                'time_value': (pair['decoded'][pair['decoded_time']].values[time_offset]
                               if pair['decoded_time'] and pair['decoded_time'] in pair['decoded'].coords else None),
                'level_value': (pair['decoded'][pair['decoded_level']].values[level_offset]
                                if pair['decoded_level'] and pair['decoded_level'] in pair['decoded'].coords else None),
            }

## 5. Remap both maps to the original challenge grid

The target coordinates exactly match `hplp_sfc_regridded_tp_025deg_steps_228_240.nc`. Healpy bilinearly interpolates the original and decoded maps at all regular-grid cell centers. Missing HEALPix values are rejected because interpolation across unknown pixels would make the relative-error result ambiguous.

In [ ]:
TARGET_LATITUDES = np.linspace(90.0, -90.0, 721, dtype=np.float64)
TARGET_LONGITUDES = np.arange(1440, dtype=np.float64) * 0.25


@lru_cache(maxsize=1)
def target_healpy_angles():
    latitude, longitude = np.meshgrid(TARGET_LATITUDES, TARGET_LONGITUDES, indexing='ij')
    return np.deg2rad(90.0 - latitude).ravel(), np.deg2rad(longitude).ravel()


def healpix_to_original_grid(values, nside, ordering, name, attrs):
    values = np.asarray(values, dtype=np.float64).squeeze()
    expected = hp.nside2npix(nside)
    if values.ndim != 1 or values.size != expected:
        raise ValueError(f'Expected one map with {expected} pixels, got {values.shape}.')
    if not np.isfinite(values).all():
        raise ValueError(f'{name!r} contains {int((~np.isfinite(values)).sum())} non-finite values.')
    ring_values = hp.reorder(values, n2r=True) if ordering == 'NESTED' else values
    theta, phi = target_healpy_angles()
    remapped = hp.get_interp_val(ring_values, theta, phi, nest=False).reshape(721, 1440)
    return xr.DataArray(
        remapped, dims=('lat', 'lon'),
        coords={'lat': TARGET_LATITUDES, 'lon': TARGET_LONGITUDES},
        name=name, attrs=dict(attrs),
    )


def remapped_relative_error(item, pair, variable, truth_order, decoded_order, attrs):
    truth_latlon = healpix_to_original_grid(
        item['truth'], pair['nside'], truth_order, variable, attrs
    )
    decoded_latlon = healpix_to_original_grid(
        item['decoded'], pair['nside'], decoded_order, variable, attrs
    )
    # Exact error-map expression from 02-relative-error-bound.ipynb.
    relative_error = xr.where(
        truth_latlon == decoded_latlon, 0,
        (decoded_latlon - truth_latlon) / np.abs(truth_latlon),
    ).assign_attrs(long_name='relative error', units='%')
    return truth_latlon, decoded_latlon, relative_error

## 6. Open and validate both stores

Only metadata is inspected initially. The cell discovers or validates variables, checks their HEALPix resolutions and selected indices, reports ordering, and shows how many 721×1440 maps will be evaluated. Confirm any warning that ordering fell back to NESTED.

In [ ]:
if not np.isfinite(RELATIVE_ERROR_BOUND) or RELATIVE_ERROR_BOUND < 0:
    raise ValueError('RELATIVE_ERROR_BOUND must be finite and non-negative.')
if TARGET_LATITUDES.shape != (721,) or TARGET_LONGITUDES.shape != (1440,):
    raise RuntimeError('The target grid must remain 721 × 1440.')
if not np.allclose(np.diff(TARGET_LATITUDES), -0.25) or not np.allclose(np.diff(TARGET_LONGITUDES), 0.25):
    raise RuntimeError('The target coordinate spacing must remain 0.25 degrees.')

ground_truth = open_zarr_store(GROUND_TRUTH_ZARR)
decoded = open_zarr_store(DECODED_ZARR)
truth_ordering = detect_ordering(ground_truth, GROUND_TRUTH_ORDERING, 'GROUND_TRUTH')
decoded_ordering = detect_ordering(decoded, DECODED_ORDERING, 'DECODED')
variable_specs = flatten_variable_configuration(VARIABLES)

if not variable_specs:
    common = []
    for name in sorted(set(ground_truth.data_vars) & set(decoded.data_vars)):
        try:
            _, truth_nside = healpix_dimension(ground_truth[name])
            _, decoded_nside = healpix_dimension(decoded[name])
        except ValueError:
            continue
        if truth_nside == decoded_nside and np.issubdtype(ground_truth[name].dtype, np.number):
            common.append(name)
    variable_specs = {name: {} for name in common}
if not variable_specs:
    raise ValueError('The stores have no common numeric HEALPix variables.')

geometry_rows = []
for variable in variable_specs:
    pair = prepare_variable_pair(
        ground_truth, decoded, variable, variable_specs[variable], VARIABLES.get('timesteps')
    )
    geometry_rows.append({
        'variable': variable, 'HPX level': int(np.log2(pair['nside'])), 'NSIDE': pair['nside'],
        'selected timesteps': len(pair['time_positions']), 'selected levels': len(pair['level_positions']),
        'maps': len(pair['time_positions']) * len(pair['level_positions']),
    })
geometry = pd.DataFrame(geometry_rows).set_index('variable')
display(pd.Series({
    'Ground truth': str(GROUND_TRUTH_ZARR), 'Decoded reconstruction': str(DECODED_ZARR),
    'Ground-truth ordering': truth_ordering, 'Decoded ordering': decoded_ordering,
    'Target grid': '721 lat × 1440 lon (0.25°)',
    'Relative error bound': f'{RELATIVE_ERROR_BOUND:.2%}',
    'Total maps': int(geometry['maps'].sum()),
}).to_frame('value'))
display(geometry)

## 7. Remap and evaluate the relative-error bound

Each selected map pair is remapped and evaluated independently. The pass/fail expression is copied from the challenge: `abs(decoded - truth) <= abs(truth) * bound`. Metrics count every latitude–longitude cell equally, including the duplicated longitude samples at the two poles, exactly as the original regular-grid analysis does. Relative errors may be infinite when a nonzero reconstruction corresponds to zero truth; those cells are violations and are counted separately.

In [ ]:
metric_rows = []
for variable in tqdm(list(variable_specs), desc='Variables'):
    pair = prepare_variable_pair(
        ground_truth, decoded, variable, variable_specs[variable], VARIABLES.get('timesteps')
    )
    map_count = len(pair['time_positions']) * len(pair['level_positions'])
    attrs = dict(decoded[variable].attrs)
    for item in tqdm(iter_aligned_maps(pair), total=map_count, desc=f'{variable} maps', leave=False):
        truth_latlon, decoded_latlon, relative_error = remapped_relative_error(
            item, pair, variable, truth_ordering, decoded_ordering, attrs
        )
        truth_values = np.asarray(truth_latlon.values)
        decoded_values = np.asarray(decoded_latlon.values)
        relative_values = np.asarray(relative_error.values)
        # Exact bound check from 02-relative-error-bound.ipynb, including zero preservation.
        passes = np.abs(decoded_values - truth_values) <= (np.abs(truth_values) * RELATIVE_ERROR_BOUND)
        violations = ~passes
        finite = np.isfinite(relative_values)
        finite_absolute = np.abs(relative_values[finite])
        metric_rows.append({
            'variable': variable, 'timestep_position': item['timestep_position'],
            'time_value': str(item['time_value']), 'level_position': item['level_position'],
            'level_value': str(item['level_value']), 'grid_cell_count': relative_values.size,
            'violation_count': int(violations.sum()), 'violation_fraction': float(violations.mean()),
            'zero_truth_count': int((truth_values == 0).sum()),
            'nonfinite_relative_error_count': int((~finite).sum()),
            'mean_absolute_relative_error': float(finite_absolute.mean()) if finite_absolute.size else np.nan,
            'max_absolute_relative_error': float(finite_absolute.max()) if finite_absolute.size else np.nan,
            'truth_min': float(truth_values.min()), 'truth_max': float(truth_values.max()),
            'decoded_min': float(decoded_values.min()), 'decoded_max': float(decoded_values.max()),
        })

metrics = pd.DataFrame(metric_rows)
if metrics.empty:
    raise RuntimeError('No maps were evaluated.')
total_cells = int(metrics['grid_cell_count'].sum())
total_violations = int(metrics['violation_count'].sum())
summary = pd.Series({
    'Result': 'PASS' if total_violations == 0 else 'FAIL',
    'Evaluated maps': len(metrics), 'Evaluated target-grid cells': total_cells,
    'Violating cells': total_violations, 'Violation fraction': total_violations / total_cells,
    'Relative error bound': RELATIVE_ERROR_BOUND,
})
display(summary.to_frame('value'))
display(metrics)

## 8. Plot one remapped comparison

These controls affect only this figure. The selected pair is remapped again, allowing the full evaluation to remain streaming. The original and decoded panels share their complete value range. The signed relative-error panel uses the challenge's fixed $\pm\epsilon_{rel}$ color range; larger violations are clipped visually but remain included in the metrics.

In [ ]:
PLOT_VARIABLE = next(iter(variable_specs))
PLOT_TIMESTEP_INDEX = 0
PLOT_LEVEL_INDEX = 0

if PLOT_VARIABLE not in variable_specs:
    raise ValueError(f'PLOT_VARIABLE must be one of {list(variable_specs)}.')
plot_specification = dict(variable_specs[PLOT_VARIABLE])
plot_cell, _ = healpix_dimension(decoded[PLOT_VARIABLE])
plot_time = axis_dimension(decoded[PLOT_VARIABLE], 'time', {plot_cell})
plot_level = axis_dimension(decoded[PLOT_VARIABLE], 'level', {plot_cell, plot_time})
if plot_level is not None:
    plot_specification['level_indices'] = [PLOT_LEVEL_INDEX]
elif PLOT_LEVEL_INDEX != 0:
    raise IndexError(f'{PLOT_VARIABLE!r} is 2-D; PLOT_LEVEL_INDEX must be 0.')
plot_pair = prepare_variable_pair(
    ground_truth, decoded, PLOT_VARIABLE, plot_specification, [PLOT_TIMESTEP_INDEX]
)
plot_item = next(iter_aligned_maps(plot_pair))
truth_latlon, decoded_latlon, relative_error = remapped_relative_error(
    plot_item, plot_pair, PLOT_VARIABLE, truth_ordering, decoded_ordering,
    dict(decoded[PLOT_VARIABLE].attrs),
)
plot_passes = np.abs(decoded_latlon.values - truth_latlon.values) <= (
    np.abs(truth_latlon.values) * RELATIVE_ERROR_BOUND
)
combined = np.concatenate((truth_latlon.values.ravel(), decoded_latlon.values.ravel()))
value_min, value_max = float(np.min(combined)), float(np.max(combined))
if value_min == value_max:
    value_max = value_min + np.finfo(float).eps
units = decoded[PLOT_VARIABLE].attrs.get('units', '')
extent = [0.0, 360.0, -90.0, 90.0]

figure, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)
panels = (
    (truth_latlon.values, 'Original', 'viridis', value_min, value_max, units),
    (decoded_latlon.values, f'Decoded with {(~plot_passes).mean():.2%} violations',
     'viridis', value_min, value_max, units),
    (relative_error.values, 'Relative compression error', 'RdBu_r',
     -RELATIVE_ERROR_BOUND, RELATIVE_ERROR_BOUND, '%'),
)
for axis, (values, title, cmap, lower, upper, label) in zip(axes, panels):
    image = axis.imshow(values, origin='upper', extent=extent, aspect='auto', cmap=cmap,
                        vmin=lower, vmax=upper, interpolation='nearest')
    axis.set_title(title)
    axis.set_xlabel('Longitude (°E)')
    axis.set_ylabel('Latitude (°N)')
    figure.colorbar(image, ax=axis, orientation='horizontal', pad=0.12, label=label)
figure.suptitle(
    f'{PLOT_VARIABLE} · decoded timestep {PLOT_TIMESTEP_INDEX} · level {PLOT_LEVEL_INDEX}', y=1.04
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_PATH = OUTPUT_DIR / f'{PLOT_VARIABLE}_latlon_relative_error.png'
figure.savefig(FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()
print(f'Figure: {FIGURE_PATH.resolve()}')

## 9. Save reusable results

The final cell stores per-map metrics and a compact JSON summary. The remapped arrays are temporary evaluation products and are not written; neither input Zarr store is modified.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = OUTPUT_DIR / 'latlon_relative_error_metrics.csv'
SUMMARY_PATH = OUTPUT_DIR / 'latlon_relative_error_summary.json'
metrics.to_csv(METRICS_PATH, index=False)
summary_payload = {
    'ground_truth_zarr': str(GROUND_TRUTH_ZARR), 'decoded_zarr': str(DECODED_ZARR),
    'variables': list(variable_specs), 'ground_truth_ordering': truth_ordering,
    'decoded_ordering': decoded_ordering, 'target_latitudes': 721, 'target_longitudes': 1440,
    'target_spacing_degrees': 0.25, 'relative_error_bound': float(RELATIVE_ERROR_BOUND),
    'zero_rule': 'truth zero passes only when decoded equals truth exactly',
    'result': summary['Result'], 'evaluated_maps': int(summary['Evaluated maps']),
    'evaluated_target_grid_cells': total_cells, 'violating_cells': total_violations,
    'violation_fraction': float(total_violations / total_cells),
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2) + '\n', encoding='utf-8')
display(pd.Series({
    'Metrics CSV': str(METRICS_PATH.resolve()),
    'Summary JSON': str(SUMMARY_PATH.resolve()),
    'Comparison figure': str(FIGURE_PATH.resolve()),
}).to_frame('path'))
ground_truth.close()
decoded.close()

## Interpretation

A `PASS` means every evaluated cell on the original 721×1440 grid satisfies the 1% pointwise relative-error bound. The metric intentionally weights every regular latitude–longitude cell equally, as the challenge does, rather than applying area weights. Near-zero truth values make a relative constraint particularly strict, while exact zeros require exact reconstruction after remapping.